# Cost Inputs Exploration

Compare the technology costs currently used in `pypsa-earth/data/costs.csv` (mostly DIW 2010 / IEA 2010–2011 vintage, in **2013 EUR**) against DEA 2030 projections from our archived `tech_config_ammonia_plant_2030_dea.yaml` (in **2020 EUR**).

**TODO:** Decide which cost inputs to use for final runs. Options:
1. Keep current costs.csv as-is (legacy PyPSA-Earth defaults, 2013 EUR)
2. Update costs.csv to DEA 2030 values (converted to 2013 EUR for consistency)
3. Update costs.csv to DEA 2030 values in 2020 EUR and adjust the model's currency year

Key question: do we want internal consistency with upstream PyPSA-Earth, or accuracy relative to our DEA reference case?

In [ ]:
import pandas as pd
import yaml
from pathlib import Path
from IPython.display import display, HTML

## Load current model costs (costs.csv)

In [ ]:
costs_csv = pd.read_csv("../pypsa-earth/data/costs.csv")

# Pivot to get investment, FOM, VOM, lifetime, efficiency, fuel per technology
costs_wide = costs_csv.pivot_table(
    index="technology", columns="parameter", values="value", aggfunc="first"
)

# Focus on the key technologies we want to compare
key_techs = [
    "solar", "onwind", "offwind", "nuclear", "OCGT", "CCGT",
    "electrolysis", "fuel cell", "hydrogen storage tank",
    "battery storage", "battery inverter",
    "ammonia synthesis", "ammonia storage",
    "CCGT H2", "CCGT NH3", "NH3 pipeline", "H2 pipeline",
]
available_techs = [t for t in key_techs if t in costs_wide.index]

cols_of_interest = ["investment", "FOM", "VOM", "lifetime", "efficiency", "fuel"]
cols_available = [c for c in cols_of_interest if c in costs_wide.columns]

current_costs = costs_wide.loc[available_techs, cols_available].copy()

# Get sources/currency year from raw CSV
inv_rows = costs_csv[
    (costs_csv["technology"].isin(available_techs)) & (costs_csv["parameter"] == "investment")
][["technology", "source", "currency_year"]].set_index("technology")

current_costs["source"] = inv_rows["source"]
current_costs["currency_year"] = inv_rows["currency_year"]

print("=== Current Model Costs (costs.csv) ===")
print("Note: investment is in EUR/kW (generators/links) or EUR/kWh (stores)")
print("All monetary values are in the currency_year shown (mostly 2013 EUR)\n")
display(current_costs.style.format(precision=2, na_rep="—"))

## Load DEA 2030 costs (archived tech config)

In [ ]:
with open("../archive/tech_config_ammonia_plant_2030_dea.yaml") as f:
    dea_config = yaml.safe_load(f)

# Extract DEA costs into a comparable DataFrame
dea_rows = []
for tech_name, tech_data in dea_config["techs"].items():
    ctype = tech_data.get("component_type", "")
    if ctype == "generator" or ctype == "link":
        cost_key = "overnight_cost_per_mw"
        unit = "EUR/kW"
        divisor = 1000  # Convert EUR/MW to EUR/kW
    elif ctype == "store":
        cost_key = "overnight_cost_per_mwh"
        unit = "EUR/kWh"
        divisor = 1000
    else:
        continue
    
    # Skip penalty/placeholder components
    if "penalty" in tech_name:
        continue
    
    inv = tech_data.get(cost_key, 0) / divisor
    dea_rows.append({
        "dea_tech": tech_name,
        "component_type": ctype,
        "investment_2020EUR": inv,
        "unit": unit,
        "lifetime": tech_data.get("lifetime_years"),
        "FOM_pct": tech_data.get("fixed_om_fraction", 0) * 100,
        "efficiency": tech_data.get("overall_efficiency"),
        "source": tech_data.get("source_note", ""),
    })

# Add DEA 2030 technologies not in the archived YAML
# CCGT: DEA "Gas turb. CC, steam extract." — 882.6 EUR/kW, eff 0.61, lifetime 25y, FOM 3.4%
dea_rows.append({
    "dea_tech": "ccgt_gas",
    "component_type": "generator",
    "investment_2020EUR": 882.6,  # EUR/kW (already in kW basis)
    "unit": "EUR/kW",
    "lifetime": 25,
    "FOM_pct": 3.4,
    "efficiency": 0.61,
    "source": "DEA 2030 Gas turb. CC, steam extract.",
})

dea_costs = pd.DataFrame(dea_rows).set_index("dea_tech")

print("=== DEA 2030 Costs (from archived tech config + manual entries, 2020 EUR) ===")
display(dea_costs.style.format(precision=2, na_rep="—"))

## Side-by-side comparison: Current vs DEA 2030

Map DEA tech names to costs.csv tech names for direct comparison.

To deflate DEA 2020 EUR → 2013 EUR, we use Eurozone HICP cumulative inflation of ~8.4% over 2013–2020 (ECB data), i.e. deflator = 1/1.084 ≈ 0.9225.

In [ ]:
# Mapping: DEA tech name → costs.csv tech name
dea_to_csv_map = {
    "solar": "solar",
    "onshore_wind": "onwind",
    "offshore_wind_fixed": "offwind",
    "ccgt_gas": "CCGT",
    # No DEA nuclear — DIW only
    "electrolysis": "electrolysis",
    "hydrogen_fuel_cell": "fuel cell",
    "ammonia_synthesis": "ammonia synthesis",
    "battery_pcs_charge": "battery inverter",
    "battery_storage": "battery storage",
    "compressed_hydrogen_store": "hydrogen storage tank",
    "ammonia": "ammonia storage",
}

# Eurozone HICP cumulative inflation 2013→2020: ~8.4%
# Source: ECB Statistical Data Warehouse, HICP overall index
DEFLATOR_2020_TO_2013 = 1 / 1.084  # ≈ 0.9225

comparison_rows = []
for dea_name, csv_name in dea_to_csv_map.items():
    if dea_name not in dea_costs.index:
        continue
    dea_row = dea_costs.loc[dea_name]
    
    csv_inv = current_costs.loc[csv_name, "investment"] if csv_name in current_costs.index else None
    csv_lifetime = current_costs.loc[csv_name, "lifetime"] if csv_name in current_costs.index else None
    csv_source = current_costs.loc[csv_name, "source"] if csv_name in current_costs.index else "—"
    
    dea_inv_2020 = dea_row["investment_2020EUR"]
    dea_inv_2013 = dea_inv_2020 * DEFLATOR_2020_TO_2013
    
    pct_diff_vs_2020 = (
        (csv_inv - dea_inv_2020) / dea_inv_2020 * 100 if csv_inv and dea_inv_2020 else None
    )
    pct_diff_vs_2013 = (
        (csv_inv - dea_inv_2013) / dea_inv_2013 * 100 if csv_inv and dea_inv_2013 else None
    )
    
    comparison_rows.append({
        "technology": csv_name,
        "dea_tech": dea_name,
        "unit": dea_row["unit"],
        "current_csv (2013 EUR)": csv_inv,
        "DEA 2030 (2020 EUR)": dea_inv_2020,
        "DEA 2030 (→2013 EUR)": dea_inv_2013,
        "csv vs DEA_2020 (%)": pct_diff_vs_2020,
        "csv vs DEA_2013 (%)": pct_diff_vs_2013,
        "csv_source": csv_source,
    })

comp_df = pd.DataFrame(comparison_rows).set_index("technology")

print("=== Investment Cost Comparison: Current costs.csv vs DEA 2030 ===")
print("Positive % = costs.csv is MORE expensive than DEA")
print("Negative % = costs.csv is CHEAPER than DEA\n")
display(
    comp_df.style.format({
        "current_csv (2013 EUR)": "{:,.1f}",
        "DEA 2030 (2020 EUR)": "{:,.1f}",
        "DEA 2030 (→2013 EUR)": "{:,.1f}",
        "csv vs DEA_2020 (%)": "{:+.1f}%",
        "csv vs DEA_2013 (%)": "{:+.1f}%",
    }, na_rep="—")
    .map(
        lambda v: "color: green" if isinstance(v, str) and v.startswith("+") else
                  ("color: red" if isinstance(v, str) and v.startswith("-") else ""),
        subset=["csv vs DEA_2020 (%)", "csv vs DEA_2013 (%)"]
    )
)

## Lifetime comparison

In [ ]:
lifetime_rows = []
for dea_name, csv_name in dea_to_csv_map.items():
    if dea_name not in dea_costs.index:
        continue
    dea_lt = dea_costs.loc[dea_name, "lifetime"]
    csv_lt = current_costs.loc[csv_name, "lifetime"] if csv_name in current_costs.index else None
    
    lifetime_rows.append({
        "technology": csv_name,
        "csv_lifetime_yr": csv_lt,
        "dea_lifetime_yr": dea_lt,
        "difference_yr": (csv_lt - dea_lt) if csv_lt and dea_lt else None,
    })

lt_df = pd.DataFrame(lifetime_rows).set_index("technology")
print("=== Lifetime Comparison ===")
display(lt_df)

## Technologies without DEA equivalents

These technologies in our costs.csv have no DEA 2030 counterpart in the archived config:

In [ ]:
techs_with_dea = set(dea_to_csv_map.values())
techs_without_dea = [t for t in available_techs if t not in techs_with_dea]

if techs_without_dea:
    no_dea = current_costs.loc[techs_without_dea, ["investment", "lifetime", "source", "currency_year"]]
    print("=== Technologies in costs.csv with NO DEA 2030 equivalent ===")
    display(no_dea)
else:
    print("All key technologies have DEA equivalents.")

## Summary & TODO

**Key observations:**
- Solar and wind costs in costs.csv are from DIW 2010 / DEA (unclear vintage), in 2013 EUR
- Nuclear costs (6,000 EUR/kW) are from DIW 2010, well below modern European new-build (~7,400–10,500 EUR/kW)
- Electrolysis and ammonia costs were already sourced from DEA 2030
- Battery storage costs may differ significantly between DIW and DEA
- Fuel cell costs (339 EUR/kW from NREL 2009) are vastly cheaper than DEA's 2,475 EUR/kW — but fuel cells are currently disabled in our model (replaced by CCGT H2)

**TODO for final runs:**
1. Decide whether to update all costs.csv entries to DEA 2030 (deflated to 2013 EUR)
2. If updating, verify that the DEA offshore wind cost (~2,393 EUR/kW) is appropriate for AC-connected (our model uses offwind-ac and offwind-dc separately)
3. Nuclear: consider using DEA 2030 or more recent estimates (~7,400 EUR/kW) instead of DIW's 6,000 EUR/kW
4. Note that changing costs will invalidate all existing results — a full re-run would be needed
5. Consider whether 2013 EUR or 2020 EUR is the better base for reporting

## SRMC Merit Order & Carbon Break-Even

**SRMC (Short-Run Marginal Cost)** is the cost of producing one additional MWh of electricity
from an already-built plant (i.e. ignoring capital costs). It determines the **merit order** --
the sequence in which generators are dispatched, cheapest first.

    SRMC = fuel_cost / efficiency + VOM + (CO2_intensity / efficiency) * carbon_price

Without a carbon price, coal and lignite dominate on pure fuel economics.
A carbon price or CO2 mass cap reverses this by penalising high-emitting, low-efficiency plants.

### How PyPSA-Earth models CO2

The model supports **two mechanisms** (via `prepare_network.py` wildcard options):

| Wildcard | Mechanism | How it works | Our scenarios |
|----------|-----------|-------------|---------------|
| `Co2L` | **Mass cap** | Adds a `GlobalConstraint` capping total annual CO2. The solver finds the *shadow price* (endogenous EUR/tCO2). | `Co2L` (77.5 Mt), `Co2zero` (approx 0 t), `Co2nocap` (1e12 t) |
| `Ep` | **Explicit carbon price** | Adds a fixed EUR/tCO2 directly to each generator's `marginal_cost`. Set in `costs.emission_prices.co2`. | **Not currently used** |

Our scenarios use **mass caps only** -- the optimizer decides the implicit carbon price.

In [ ]:
# === SRMC Merit Order: Comparative table with carbon break-even ===
import pandas as pd

data = {
    "Carrier":           ["Lignite", "Coal",  "CCGT",  "OCGT",  "Oil",   "Nuclear"],
    "Fuel (EUR/MWhth)":  [6.0,       15.0,    40.0,    40.0,    54.2,    3.3],
    "Efficiency":        [0.447,     0.464,   0.580,   0.410,   0.393,   0.337],
    "CO2 (t/MWhth)":     [0.336,     0.336,   0.198,   0.198,   0.248,   0.0],
    "VOM (EUR/MWhel)":   [3.0,       3.5,     3.3,     3.3,     0.0,     0.0],
}
df = pd.DataFrame(data).set_index("Carrier")

# Derived columns
df["Fuel (EUR/MWhel)"] = df["Fuel (EUR/MWhth)"] / df["Efficiency"]
df["CO2 (t/MWhel)"]    = df["CO2 (t/MWhth)"] / df["Efficiency"]
df["SRMC no CO2"]       = df["Fuel (EUR/MWhel)"] + df["VOM (EUR/MWhel)"]

# Carbon price at which each fuel's total SRMC equals CCGT's total SRMC
# SRMC_i + CO2_el_i * Pc = SRMC_ccgt + CO2_el_ccgt * Pc
# => Pc = (SRMC_ccgt - SRMC_i) / (CO2_el_i - CO2_el_ccgt)
ccgt_srmc   = df.loc["CCGT", "SRMC no CO2"]
ccgt_co2_el = df.loc["CCGT", "CO2 (t/MWhel)"]
df["CO2 price to match CCGT (EUR/t)"] = (
    (ccgt_srmc - df["SRMC no CO2"]) / (df["CO2 (t/MWhel)"] - ccgt_co2_el)
).round(0)
df.loc["CCGT",    "CO2 price to match CCGT (EUR/t)"] = float("nan")
df.loc["Nuclear", "CO2 price to match CCGT (EUR/t)"] = float("nan")

# Display
cols = [
    "Fuel (EUR/MWhth)", "Efficiency", "CO2 (t/MWhth)",
    "Fuel (EUR/MWhel)", "VOM (EUR/MWhel)", "SRMC no CO2",
    "CO2 (t/MWhel)", "CO2 price to match CCGT (EUR/t)",
]
display(df[cols].round(1).style.set_caption(
    "SRMC Merit Order -- DEA 2030 fuel costs (2020 EUR), no carbon price"
).format(precision=1, na_rep="--"))

print()
print("Key: 'CO2 price to match CCGT' = carbon price at which each fuel's")
print("total SRMC (fuel + VOM + carbon) equals CCGT's total SRMC.")
print(f"\nCCGT SRMC (no CO2 price) = {ccgt_srmc:.1f} EUR/MWhel")
print("EU ETS price 2024-26: approx 60-70 EUR/tCO2 => coal & lignite uncompetitive in reality.")